![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 8</p>

---
---

**Remark**: Some text in this document was generated using Gemini.


# Table of contents for week 8
1. [Introduction to Natural Language Processing](#intro_nlp)
     1. [Automatic Speech Recognition](#asr)
     2. [Natural Language Generation](#nlg)
2. [Word, Sentence, Corpus](#wordsentencecorpus)
3. [Tokenization](#tokenization)
4. [Vocabulary Building](#vocab)
5. [Language Models](#languagemodels)
     1. [N-Grams and Markov assumption](#ngrams)
     2. [Estimation of probabilities](#pestim)
6. [Perplexity](#perplexity)
7. [Literature](#literature)


## 1. Introduction to Natural Language Processing <a name="intro_nlp"></a>
Natural Language Processing (NLP) is a subfield of computer science and especially artificial intelligence. It is primarily concerned with providing computers with the ability to process data encoded in natural language and is thus closely related to information retrieval, knowledge representation and computational linguistics, a subfield of linguistics. Typically data is collected in text corpora, using either rule-based, statistical or neural-based approaches in machine learning and deep learning.

Major tasks in natural language processing are speech recognition, text classification, natural-language understanding, and natural-language generation. [Wikipedia]

In the past few years, NLP has seen significant advancements due to the rapid development of Large Language Models (LLMs). Being considered generative AI within NLP, LLMs allow for highly sophisticated text generation, translation, and other complex language tasks, with a level of fluency previously unseen.

### 1.1. Automatic Speech Recognition <a name="asr"></a>
Automatic Speech Recognition (ASR), also known as speech-to-text, is a technology that enables a computer or device to understand and transcribe spoken language into written text.

ASR is used in a wide range of applications, including:
- Voice assistants: Siri, Alexa, Google Assistant
- Dictation software: Converting spoken words into documents
- Transcription services: Creating written transcripts of audio or video recordings
- Accessibility tools: Helping people with disabilities to interact with technology
- Customer service: Automating phone systems and chatbots

We will look into a state-of-the-art speech-to-text network in SW09.

### 1.2. Natural Language Generation <a name="nlg"></a>
Natural Language Generation (NLG) is a subfield of artificial intelligence that focuses on enabling computers to produce human-like text or speech. Essentially, it's about teaching machines to communicate with us in a way that we can easily understand.

Think of it as the opposite of ASR. With ASR, the computer understands what we say. With NLG, the computer generates its own text or speech.

#### Application: Voice Cloning

Try to clone your voice here: [coqui TTS](https://huggingface.co/spaces/coqui/xtts).


## 2. Word, Sentence, Corpus <a name="wordsentencecorpus"></a>
Obviously the <em>sentence</em> <code>"Daisy teaches English."</code> can be partitioned into three <em>words</em>. (The perion <code>"."</code> may also be treated as a word, depending on the task.)

Word order, also known as syntax, is a fundamental aspect of how we understand meaning in language. However, note that partitioning sentences into words is more difficult in some languages:

- Japanese: 私は猫が好きなので毎日猫の写真を撮ります
- Thai: ทุกคนก็พร้อมประชุมร่วมและพิจารณาร่วมกัน

In NLP, a <em>corpus</em> (plural: corpora) is a large and structured collection of text or spoken language. See [English corpora](https://www.english-corpora.org/) for examples.


## 3. Tokenization <a name="tokenization"></a>
In NLP, "tokenization" refers to the process of breaking down a piece of text into smaller, meaningful units called "tokens," which can be individual words, characters, or even sentences, allowing machines to analyze and understand the text more effectively by dividing it into manageable parts; essentially, it's the first step in most NLP tasks to prepare text for further processing.

See [[1], Chap. 2.5](#literature).

## 4. Vocabulary Building<a name="vocab"></a>
A set of tokens is normally referred to as a vocabulary. There are two widely used algorithms for token segmentation and vocabulary building: <em>byte-pair encoding</em> (see [[1], Chap. 2.5.2](#literature) for details), and <em>unigram language modeling</em>. Imagine you have a bag of words, and you want to predict the next word in a sentence. A unigram language model assumes that each word in the sentence is independent of all the other words. It only cares about how often each individual word appears in a corpus.

### Exercise: vocabulary building

The following program builds a vocabulary on the basis of a corpus. Upload some text sources (file names must end with .txt) to the sample_data folder and run the program. Understand and explain what is being done.

In [ ]:
# build vocabulary

import os
from collections import Counter
import pickle

def read_data(path,files):
    text_data = ""
    for fle in files:
        if (fle[-4:]=='.txt'):
            with open(os.path.join(path,fle), encoding='utf-8', errors="ignore") as f:
                lines = f.readlines()
                text_data += ''.join(lines)
    return text_data

def build_vocab(text_data, max_vocab=10000, min_freq=3):
    token_counter = Counter()
    for word in text_data.split():
        token = word.strip('><»«-–",.!:;?„“‚‘†').strip("'").lower()
        if (len(token)<2):
            continue
        token_counter[token] += 1

    token2id = {'<unk>':0,'<eos>':1}
    id2token = {0:'<unk>',1:'<eos>'}
    for token,count in token_counter.most_common():
        if (count<min_freq):
            break
        if (len(token2id)>=max_vocab):
            break
        tid = len(token2id)
        token2id[token] = tid
        id2token[tid] = token
    return token2id,id2token

path = "./sample_data"
data_files = os.listdir(path)
text_data = read_data(path,data_files)

token2id,id2token = build_vocab(text_data, max_vocab=1000)
print("vocab size:",len(token2id))

# save dictionaries
with open('vocabulary.pkl', 'wb') as f:
    pickle.dump((token2id,id2token), f)

In [ ]:
# list 100 most common tokens

for i in range(100):
    print(i,"[",id2token[i],"]")

## 5. Language Models <a name="languagemodels"></a>
A language model is a machine learning model that predicts upcoming words. Predictions are typically made on the basis of probabilities.

### 5.1. N-Grams and Markov assumption <a name="ngrams"></a>
Probability of a sequence of words:

$P("\mbox{The water of Walden Pond is so beautifully}")=P(w_{1:n-1})$,

where $w_{1:n-1}$ stands for $w_1,\dots,w_{n-1}$, and $n=9$.

Conditional probability of a sequence of words:

$P("\mbox{blue}"|"\mbox{The water of Walden Pond is so beautifully}")=P(w_n|w_{1:n-1})$.

Chain rule:

$P(w_{1:n})=P(w_1)P(w_2|w_1)P(w_3|w_{1:2})\cdots P(w_n|w_{1:n−1})=\prod_{k=1}^n P(w_k|w_{1:k−1})$

bigram model: $P(w_n|w_{1:n-1})\approx P(w_n|w_{n-1})$

N-gram model: $P(w_n|w_{1:n-1})\approx P(w_n|w_{n-N+1:n-1})$

Note that for building an N-gram model, one needs $|V|^N$ probabilities, where $|V|$ is the number of words.

### 5.2. Estimation of probabilities <a name="pestim"></a>
A method for estimating the above probabilities is by counting occurrences of word sequences in a corpus. Denoting the number of occurrences of the sequence $w_1,w_2$ as $C(w_1w_2)$, the following estimate may be used for building a bigram model:
$$
P(w_n|w_{n−1})=\frac{C(w_{n−1}w_n)}{\sum_w C(w_{n−1}w)}.
$$
Or, equivalently,
$$
P(w_n|w_{n−1})=\frac{C(w_{n−1}w_n)}{C(w_{n−1})}.
$$
See examples in [[1], Chap. 3.1.2](#literature).

## 6. Perplexity <a name="perplexity"></a>
In the context of language models, perplexity is a metric that measures how well a probability model predicts a sample. Essentially, it quantifies how "surprised" a model is when it encounters a piece of text.

Test set $W$ has perplexity
$$
{\rm perplexity}(W)=P(w_1w_2\dots w_N)^{-\frac{1}{N}}.
$$
The lower the perplexity of a model on the data, the better the model!

Perplexity of $W$ with bigram language model:
$$
{\rm perplexity}(W)=\left(\prod_{i=1}^N P(w_i|w_{i-1})\right)^{-\frac{1}{N}}
$$
See examples in [[1], Chap. 3.1.3](#literature).


## 7. Literature <a name="literature"></a>

<a id="ref1" href="https://web.stanford.edu/~jurafsky/slp3">[1]</a> Daniel Jurafsky and James H. Martin. 2025. Speech and Language Processing: An Introduction to Natural Language Processing, Computational Linguistics, and Speech Recognition with Language Models, 3rd edition. Online manuscript released January 12, 2025. https://web.stanford.edu/~jurafsky/slp3.
